In [5]:
import pandas as pd
import numpy as np

# Chargement des données
df = pd.read_csv("Donnes_de_recouvrement_hotel_res.csv")

# Conversion des dates
date_cols = ['date_facture', 'date_echeance', 'date_paiement', 'relance_1', 'relance_2']
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Date d'analyse (pour les paiements non faits)
today = pd.Timestamp("2025-07-21")

# Calcul des jours de retard
df["jours_de_retard"] = (df["date_paiement"].fillna(today) - df["date_echeance"]).dt.days
df["jours_de_retard"] = df["jours_de_retard"].apply(lambda x: max(x, 0))

# Calcul du montant dû
df["montant_du"] = np.where(df["statut_paiement"] == "Payé", 0, df["montant_facture"])

# Coefficient selon type
df["coeff_type"] = df["type"].apply(lambda x: 1 if x == "Chaîne" else 1.5)

# Score de priorité
df["score_priorite"] = df["montant_du"] * df["jours_de_retard"] * df["coeff_type"]

# Définir le seuil de retard (modifiable)
seuil_retard = 30

# Sélection des hôtels avec retard > seuil
hotels_en_retard = df[df["jours_de_retard"] > seuil_retard].copy()

# Export des données enrichies
df.to_csv("donnees_recouvrement_avec_scores.csv", index=False)

# Export des hôtels en fort retard
hotels_en_retard.to_csv("hotels_en_retard_plus_30j.csv", index=False)

# Export du top 10 des hôtels à relancer
top_10_retards = hotels_en_retard.sort_values("score_priorite", ascending=False).head(10)
top_10_retards.to_csv("top_10_hotels_a_relancer.csv", index=False)
